# ML HW2 - Clustering 분석
## 서울시 상권분석서비스 데이터 기반 상권 군집화

**데이터 출처**: 서울 열린데이터광장  
**분석 단위**: 서울시 상권 (1,581개)  
**Label (y)**: 상권_구분_코드_명 (골목상권 / 발달상권 / 전통시장 / 관광특구)

## 0. 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
matplotlib.rcParams['font.family'] = 'Malgun Gothic'  # Windows
# matplotlib.rcParams['font.family'] = 'AppleGothic'  # Mac
matplotlib.rcParams['axes.unicode_minus'] = False

import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, AffinityPropagation
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist

print('라이브러리 로드 완료')

---
## 1. 데이터 수집 및 이해

### 1-1. 데이터 로드

In [ ]:
# ※ 파일 경로를 실제 위치에 맞게 수정하세요
SALES_PATH = '서울시_상권분석서비스_추정매출-상권__2024년.csv'
STORE_PATH = '서울시_상권분석서비스_점포-상권__2024년.csv'
WORK_PATH  = '서울시_상권분석서비스_직장인구-상권_.csv'

df_sales = pd.read_csv(SALES_PATH, encoding='cp949')
df_store = pd.read_csv(STORE_PATH, encoding='cp949')
df_work  = pd.read_csv(WORK_PATH,  encoding='cp949')

print(f'추정매출 shape: {df_sales.shape}')
print(f'점포      shape: {df_store.shape}')
print(f'직장인구  shape: {df_work.shape}')

### 1-2. 데이터 이해

**[추정매출-상권]** 55개 컬럼, 87,179행 (상권 × 업종 단위)
- 당월 매출금액/건수, 주중·주말, 요일별(7), 시간대별(6), 성별(2), 연령대별(6) 매출

**[점포-상권]** 14개 컬럼, 306,889행 (상권 × 업종 단위)
- 점포수, 유사업종점포수, 개업률·개업점포수, 폐업률·폐업점포수, 프랜차이즈점포수

**[직장인구-상권]** 26개 컬럼, 45,840행 (상권 × 분기 단위)
- 총직장인구, 남/여, 연령대별(6), 성별×연령대별(12) 직장인구

**분석 전략**: 2024년 4분기(20244) 기준으로 필터 후, 상권 단위로 집계하여 1,581개 상권을 관측치로 분석

In [ ]:
# 분기별 분포 확인
TARGET_Q = 20244  # 2024년 4분기

print('[상권 구분 분포]')
print(df_sales[df_sales['기준_년분기_코드'] == TARGET_Q]
      .groupby('상권_구분_코드_명')['상권_코드'].nunique())

df_sales.head(3)

---
## 2. 전처리

### 2-1. 분기 필터 및 상권 단위 집계

In [ ]:
# ── 추정매출: 2024Q4 필터 후 상권별 sum ──────────────────────────
sales_num_cols = [
    '당월_매출_금액', '당월_매출_건수',
    '주중_매출_금액', '주말_매출_금액',
    '월요일_매출_금액', '화요일_매출_금액', '수요일_매출_금액',
    '목요일_매출_금액', '금요일_매출_금액', '토요일_매출_금액', '일요일_매출_금액',
    '시간대_00~06_매출_금액', '시간대_06~11_매출_금액', '시간대_11~14_매출_금액',
    '시간대_14~17_매출_금액', '시간대_17~21_매출_금액', '시간대_21~24_매출_금액',
    '남성_매출_금액', '여성_매출_금액',
    '연령대_10_매출_금액', '연령대_20_매출_금액', '연령대_30_매출_금액',
    '연령대_40_매출_금액', '연령대_50_매출_금액', '연령대_60_이상_매출_금액'
]

sales_q = df_sales[df_sales['기준_년분기_코드'] == TARGET_Q].copy()
sales_agg = (
    sales_q.groupby(['상권_코드', '상권_코드_명', '상권_구분_코드_명'])[sales_num_cols]
    .sum()
    .reset_index()
)

# ── 점포: 2024Q4 필터 후 상권별 sum ───────────────────────────────
store_num_cols = ['점포_수', '유사_업종_점포_수', '개업_점포_수', '폐업_점포_수', '프랜차이즈_점포_수']

store_q = df_store[df_store['기준_년분기_코드'] == TARGET_Q].copy()
store_agg = (
    store_q.groupby('상권_코드')[store_num_cols]
    .sum()
    .reset_index()
)

# ── 직장인구: 2024Q4 필터 ─────────────────────────────────────────
work_num_cols = [
    '총_직장_인구_수', '남성_직장_인구_수', '여성_직장_인구_수',
    '연령대_10_직장_인구_수', '연령대_20_직장_인구_수', '연령대_30_직장_인구_수',
    '연령대_40_직장_인구_수', '연령대_50_직장_인구_수', '연령대_60_이상_직장_인구_수'
]

work_q = df_work[df_work['기준_년분기_코드'] == TARGET_Q][['상권_코드'] + work_num_cols].copy()

print(f'매출 집계: {sales_agg.shape}, 점포 집계: {store_agg.shape}, 직장인구: {work_q.shape}')

### 2-2. 비율 변수 생성 (상권 규모 정규화)

In [ ]:
# 절대금액 대신 '비율 변수'를 사용 → 상권 크기 편차 제거
eps = 1  # 0 나누기 방지

# 주중/주말 비율
sales_agg['주중_매출_비율'] = sales_agg['주중_매출_금액'] / (sales_agg['당월_매출_금액'] + eps)
sales_agg['주말_매출_비율'] = sales_agg['주말_매출_금액'] / (sales_agg['당월_매출_금액'] + eps)

# 요일별 비율
for day in ['월요일', '화요일', '수요일', '목요일', '금요일', '토요일', '일요일']:
    sales_agg[f'{day}_매출_비율'] = sales_agg[f'{day}_매출_금액'] / (sales_agg['당월_매출_금액'] + eps)

# 시간대별 비율
for t in ['00~06', '06~11', '11~14', '14~17', '17~21', '21~24']:
    sales_agg[f'시간대_{t}_비율'] = sales_agg[f'시간대_{t}_매출_금액'] / (sales_agg['당월_매출_금액'] + eps)

# 성별 비율
sales_agg['여성_매출_비율'] = sales_agg['여성_매출_금액'] / (sales_agg['당월_매출_금액'] + eps)

# 연령대별 비율
for age in ['10', '20', '30', '40', '50', '60_이상']:
    sales_agg[f'연령대_{age}_비율'] = sales_agg[f'연령대_{age}_매출_금액'] / (sales_agg['당월_매출_금액'] + eps)

# 프랜차이즈 비율 (점포 대비)
store_agg['프랜차이즈_비율'] = store_agg['프랜차이즈_점포_수'] / (store_agg['점포_수'] + eps)
store_agg['개업_비율']       = store_agg['개업_점포_수'] / (store_agg['점포_수'] + eps)
store_agg['폐업_비율']       = store_agg['폐업_점포_수'] / (store_agg['점포_수'] + eps)

# 직장인구 성별 비율
work_q = work_q.copy()
work_q['직장_여성_비율'] = work_q['여성_직장_인구_수'] / (work_q['총_직장_인구_수'] + eps)
work_q['직장_2030_비율'] = (work_q['연령대_20_직장_인구_수'] + work_q['연령대_30_직장_인구_수']) / (work_q['총_직장_인구_수'] + eps)

print('비율 변수 생성 완료')

### 2-3. 세 데이터 병합

In [ ]:
# merge
df = sales_agg.merge(store_agg, on='상권_코드', how='left')
df = df.merge(work_q, on='상권_코드', how='left')

print(f'병합 후 shape: {df.shape}')
print(f'결측값:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

### 2-4. 분석용 feature 선택 및 결측치·이상치 처리

In [ ]:
# 분석에 사용할 feature 목록 (총 28개)
FEATURE_COLS = [
    # 매출 규모 (log 변환 예정)
    '당월_매출_금액',
    # 주중/주말
    '주중_매출_비율', '주말_매출_비율',
    # 요일별 비율
    '월요일_매출_비율', '화요일_매출_비율', '수요일_매출_비율',
    '목요일_매출_비율', '금요일_매출_비율', '토요일_매출_비율', '일요일_매출_비율',
    # 시간대별 비율
    '시간대_00~06_비율', '시간대_06~11_비율', '시간대_11~14_비율',
    '시간대_14~17_비율', '시간대_17~21_비율', '시간대_21~24_비율',
    # 성별
    '여성_매출_비율',
    # 연령대
    '연령대_10_비율', '연령대_20_비율', '연령대_30_비율',
    '연령대_40_비율', '연령대_50_비율', '연령대_60_이상_비율',
    # 점포
    '점포_수', '프랜차이즈_비율', '개업_비율', '폐업_비율',
    # 직장인구
    '총_직장_인구_수', '직장_여성_비율', '직장_2030_비율'
]

df_feat = df[['상권_코드', '상권_코드_명', '상권_구분_코드_명'] + FEATURE_COLS].copy()

# 결측치 처리: 직장인구 없는 상권은 0으로 채움
df_feat[FEATURE_COLS] = df_feat[FEATURE_COLS].fillna(0)

# 매출 0인 상권 제거 (데이터 없는 상권)
df_feat = df_feat[df_feat['당월_매출_금액'] > 0].reset_index(drop=True)

# 매출 금액 log 변환 (우편포를 정규분포에 가깝게)
df_feat['당월_매출_금액'] = np.log1p(df_feat['당월_매출_금액'])
df_feat['점포_수']        = np.log1p(df_feat['점포_수'])
df_feat['총_직장_인구_수'] = np.log1p(df_feat['총_직장_인구_수'])

print(f'최종 분석 데이터: {df_feat.shape}')
print(f'feature 수: {len(FEATURE_COLS)}개')
df_feat[FEATURE_COLS].describe().round(3)

### 2-5. 스케일링 및 Label 분리

In [ ]:
# Label 저장 (군집화 입력에는 사용 안 함 - external evaluation 용)
y_true_str = df_feat['상권_구분_코드_명'].values
le = LabelEncoder()
y_true = le.fit_transform(y_true_str)
print('클래스 매핑:', dict(zip(le.classes_, le.transform(le.classes_))))

# StandardScaler 적용
scaler = StandardScaler()
X = scaler.fit_transform(df_feat[FEATURE_COLS])

print(f'\nX shape: {X.shape}')
print(f'평균(첫 3개 feature): {X.mean(axis=0)[:3].round(3)}')
print(f'표준편차(첫 3개 feature): {X.std(axis=0)[:3].round(3)}')

---
## 3. 군집화 분석

### 3-1. K-Means

In [ ]:
# ── Elbow Method (SSE) ────────────────────────────────────────────
sse = []
sil = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    sse.append(km.inertia_)
    sil.append(silhouette_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(K_range, sse, 'bo-')
axes[0].set_xlabel('K (군집 수)')
axes[0].set_ylabel('SSE (Inertia)')
axes[0].set_title('Elbow Method')
axes[0].grid(True)

axes[1].plot(K_range, sil, 'rs-')
axes[1].set_xlabel('K (군집 수)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')
axes[1].grid(True)

plt.tight_layout()
plt.savefig('plot_kmeans_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

print('K별 Silhouette Score:')
for k, s in zip(K_range, sil):
    print(f'  K={k}: {s:.4f}')

In [ ]:
# ── 최적 K로 최종 K-Means ─────────────────────────────────────────
BEST_K = 4  # Elbow/Silhouette 결과 보고 조정 (label 클래스 수와 맞춤)

km_best = KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
km_labels = km_best.fit_predict(X)

df_feat['kmeans_label'] = km_labels

print(f'K-Means (K={BEST_K}) 결과')
print(f'  SSE: {km_best.inertia_:.2f}')
print(f'  Silhouette Score: {silhouette_score(X, km_labels):.4f}')
print(f'  군집별 크기:\n{pd.Series(km_labels).value_counts().sort_index()}')

### 3-2. Hierarchical Clustering (계층적 군집화)

In [ ]:
# ── Dendrogram (샘플 200개로 시각화) ─────────────────────────────
from sklearn.utils import resample
idx_sample = resample(range(len(X)), n_samples=200, random_state=42)
X_sample = X[idx_sample]

linked = linkage(X_sample, method='ward')

plt.figure(figsize=(14, 5))
dendrogram(linked, truncate_mode='lastp', p=30,
           leaf_rotation=90, leaf_font_size=10, show_contracted=True)
plt.title('Hierarchical Clustering Dendrogram (Ward linkage, 샘플 200개)')
plt.xlabel('상권 (클러스터 크기)')
plt.ylabel('Distance')
plt.tight_layout()
plt.savefig('plot_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── linkage 방법별 Silhouette 비교 ────────────────────────────────
linkage_methods = ['ward', 'complete', 'average']
hier_results = {}

for method in linkage_methods:
    hc = AgglomerativeClustering(n_clusters=BEST_K, linkage=method)
    labels = hc.fit_predict(X)
    sil = silhouette_score(X, labels)
    hier_results[method] = (labels, sil)
    print(f'  Hierarchical ({method:8s}): Silhouette={sil:.4f}')

# 최적 linkage 선택
best_method = max(hier_results, key=lambda m: hier_results[m][1])
hier_labels = hier_results[best_method][0]
df_feat['hier_label'] = hier_labels
print(f'\n채택 linkage: {best_method} (Silhouette={hier_results[best_method][1]:.4f})')

### 3-3. DBSCAN

In [ ]:
# ── k-dist graph로 eps 탐색 ───────────────────────────────────────
from sklearn.neighbors import NearestNeighbors

k_nn = 5
nbrs = NearestNeighbors(n_neighbors=k_nn).fit(X)
distances, _ = nbrs.kneighbors(X)
k_distances = np.sort(distances[:, k_nn-1])[::-1]

plt.figure(figsize=(8, 4))
plt.plot(k_distances)
plt.xlabel('포인트 (정렬)')
plt.ylabel(f'{k_nn}-NN 거리')
plt.title('k-Distance Graph (eps 선택 참고)')
plt.grid(True)
plt.tight_layout()
plt.savefig('plot_kdistance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── eps / min_samples 그리드 탐색 ─────────────────────────────────
eps_list = [1.5, 2.0, 2.5, 3.0, 3.5]
min_samples_list = [3, 5, 10]

dbscan_results = []
for eps in eps_list:
    for ms in min_samples_list:
        db = DBSCAN(eps=eps, min_samples=ms)
        labels = db.fit_predict(X)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        noise_ratio = (labels == -1).sum() / len(labels)
        sil = silhouette_score(X, labels) if n_clusters > 1 else -1
        dbscan_results.append({'eps': eps, 'min_samples': ms,
                               'n_clusters': n_clusters,
                               'noise_ratio': round(noise_ratio, 3),
                               'silhouette': round(sil, 4)})

dbscan_df = pd.DataFrame(dbscan_results)
print(dbscan_df.to_string(index=False))

In [ ]:
# ── 최적 파라미터로 DBSCAN ───────────────────────────────────────
# 위 결과 보고 n_clusters가 2~6 사이이면서 noise_ratio 낮은 조합 선택
BEST_EPS = 2.5    # 결과 보고 수정
BEST_MS  = 5      # 결과 보고 수정

db_best = DBSCAN(eps=BEST_EPS, min_samples=BEST_MS)
db_labels = db_best.fit_predict(X)
n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)

df_feat['dbscan_label'] = db_labels

print(f'DBSCAN (eps={BEST_EPS}, min_samples={BEST_MS})')
print(f'  군집 수: {n_db_clusters}')
print(f'  노이즈 비율: {(db_labels==-1).sum()/len(db_labels):.3f}')
if n_db_clusters > 1:
    print(f'  Silhouette Score: {silhouette_score(X, db_labels):.4f}')
print(f'  군집별 크기:\n{pd.Series(db_labels).value_counts().sort_index()}')

### 3-4. Affinity Propagation

In [ ]:
# ── damping 파라미터 탐색 ─────────────────────────────────────────
# ※ 데이터가 많으면 오래 걸릴 수 있음 → 500개 샘플로 탐색
from sklearn.utils import resample as sk_resample
X_ap = sk_resample(X, n_samples=min(500, len(X)), random_state=42)

damping_list = [0.5, 0.6, 0.7, 0.8, 0.9]
ap_results = []

for d in damping_list:
    ap = AffinityPropagation(damping=d, random_state=42, max_iter=300)
    labels = ap.fit_predict(X_ap)
    n_cl = len(set(labels))
    sil = silhouette_score(X_ap, labels) if n_cl > 1 else -1
    ap_results.append({'damping': d, 'n_clusters': n_cl, 'silhouette': round(sil, 4)})

ap_df = pd.DataFrame(ap_results)
print(ap_df.to_string(index=False))

In [ ]:
# ── 전체 데이터 Affinity Propagation ────────────────────────────
BEST_DAMPING = 0.9   # 결과 보고 수정 (군집 수 너무 많으면 damping 높임)

ap_best = AffinityPropagation(damping=BEST_DAMPING, random_state=42, max_iter=500)
ap_labels = ap_best.fit_predict(X)

df_feat['ap_label'] = ap_labels

n_ap = len(set(ap_labels))
sil_ap = silhouette_score(X, ap_labels) if n_ap > 1 else -1
print(f'Affinity Propagation (damping={BEST_DAMPING})')
print(f'  군집 수: {n_ap}')
print(f'  Silhouette Score: {sil_ap:.4f}')

### 3-5. 알고리즘별 성능 비교

In [ ]:
results_summary = pd.DataFrame([
    {
        'Algorithm': 'K-Means',
        'Hyperparameter': f'K={BEST_K}',
        'n_clusters': BEST_K,
        'SSE': round(km_best.inertia_, 2),
        'Silhouette': round(silhouette_score(X, km_labels), 4)
    },
    {
        'Algorithm': 'Hierarchical',
        'Hyperparameter': f'linkage={best_method}',
        'n_clusters': BEST_K,
        'SSE': '-',
        'Silhouette': round(hier_results[best_method][1], 4)
    },
    {
        'Algorithm': 'DBSCAN',
        'Hyperparameter': f'eps={BEST_EPS}, min_samples={BEST_MS}',
        'n_clusters': n_db_clusters,
        'SSE': '-',
        'Silhouette': round(silhouette_score(X, db_labels), 4) if n_db_clusters > 1 else '-'
    },
    {
        'Algorithm': 'Affinity Propagation',
        'Hyperparameter': f'damping={BEST_DAMPING}',
        'n_clusters': n_ap,
        'SSE': '-',
        'Silhouette': round(sil_ap, 4)
    }
])

print('=== 알고리즘별 군집화 성능 비교 ===')
print(results_summary.to_string(index=False))

---
## 4. 군집 결과 분석 및 시각화

### 4-1. PCA 2D 시각화

In [ ]:
# PCA 2D
pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X)

print(f'PCA 2D 설명 분산 비율: {pca2.explained_variance_ratio_}')
print(f'누적 설명 분산: {pca2.explained_variance_ratio_.sum():.4f}')

# 상권 구분별 색상
label_colors = {'골목상권': '#4C72B0', '발달상권': '#DD8452', '전통시장': '#55A868', '관광특구': '#C44E52'}
color_map = [label_colors.get(l, 'gray') for l in y_true_str]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 실제 label
for label_name, color in label_colors.items():
    mask = y_true_str == label_name
    axes[0].scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                    c=color, label=label_name, alpha=0.6, s=20)
axes[0].set_title('PCA 2D - 실제 상권 구분 (label)')
axes[0].legend()
axes[0].set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.2%})')
axes[0].set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.2%})')

# K-Means 군집 결과
scatter = axes[1].scatter(X_pca2[:, 0], X_pca2[:, 1],
                          c=km_labels, cmap='tab10', alpha=0.6, s=20)
axes[1].set_title(f'PCA 2D - K-Means (K={BEST_K})')
plt.colorbar(scatter, ax=axes[1])
axes[1].set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.2%})')
axes[1].set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.2%})')

plt.tight_layout()
plt.savefig('plot_pca2d.png', dpi=150, bbox_inches='tight')
plt.show()

### 4-2. PCA 3D 시각화

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

pca3 = PCA(n_components=3, random_state=42)
X_pca3 = pca3.fit_transform(X)
print(f'PCA 3D 누적 설명 분산: {pca3.explained_variance_ratio_.sum():.4f}')

fig = plt.figure(figsize=(14, 5))

# 실제 label
ax1 = fig.add_subplot(121, projection='3d')
for label_name, color in label_colors.items():
    mask = y_true_str == label_name
    ax1.scatter(X_pca3[mask, 0], X_pca3[mask, 1], X_pca3[mask, 2],
                c=color, label=label_name, alpha=0.5, s=15)
ax1.set_title('PCA 3D - 실제 label')
ax1.legend(fontsize=8)

# K-Means
ax2 = fig.add_subplot(122, projection='3d')
sc = ax2.scatter(X_pca3[:, 0], X_pca3[:, 1], X_pca3[:, 2],
                 c=km_labels, cmap='tab10', alpha=0.5, s=15)
ax2.set_title(f'PCA 3D - K-Means (K={BEST_K})')

plt.tight_layout()
plt.savefig('plot_pca3d.png', dpi=150, bbox_inches='tight')
plt.show()

### 4-3. t-SNE 시각화

In [ ]:
# t-SNE (시간 걸릴 수 있음 ~1-2분)
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 실제 label
for label_name, color in label_colors.items():
    mask = y_true_str == label_name
    axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    c=color, label=label_name, alpha=0.6, s=20)
axes[0].set_title('t-SNE - 실제 상권 구분 (label)')
axes[0].legend()

# K-Means
sc1 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=km_labels, cmap='tab10', alpha=0.6, s=20)
axes[1].set_title(f't-SNE - K-Means (K={BEST_K})')
plt.colorbar(sc1, ax=axes[1])

# Hierarchical
sc2 = axes[2].scatter(X_tsne[:, 0], X_tsne[:, 1], c=hier_labels, cmap='tab10', alpha=0.6, s=20)
axes[2].set_title(f't-SNE - Hierarchical ({best_method})')
plt.colorbar(sc2, ax=axes[2])

plt.tight_layout()
plt.savefig('plot_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

### 4-4. 전체 알고리즘 군집 결과 비교 (PCA 2D)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

plot_data = [
    ('실제 label', y_true_str, None),
    (f'K-Means (K={BEST_K})', km_labels, 'tab10'),
    (f'Hierarchical ({best_method})', hier_labels, 'tab10'),
    (f'DBSCAN (eps={BEST_EPS})', db_labels, 'tab10'),
    (f'Affinity Propagation (d={BEST_DAMPING})', ap_labels, 'tab10'),
]

for i, (title, labels, cmap) in enumerate(plot_data):
    if title == '실제 label':
        for label_name, color in label_colors.items():
            mask = y_true_str == label_name
            axes[i].scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                            c=color, label=label_name, alpha=0.5, s=15)
        axes[i].legend(fontsize=8)
    else:
        sc = axes[i].scatter(X_pca2[:, 0], X_pca2[:, 1],
                             c=labels, cmap=cmap, alpha=0.5, s=15)
        plt.colorbar(sc, ax=axes[i])
    axes[i].set_title(title)
    axes[i].set_xlabel('PC1')
    axes[i].set_ylabel('PC2')

axes[-1].axis('off')

plt.suptitle('군집화 알고리즘별 결과 비교 (PCA 2D)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('plot_all_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 4-5. K-Means 군집 특성 분석 (실제 label과 비교)

In [ ]:
# 군집별 실제 label 분포
print('=== K-Means 군집별 실제 상권 구분 분포 ===')
cross_tab = pd.crosstab(df_feat['kmeans_label'], df_feat['상권_구분_코드_명'])
print(cross_tab)

# 히트맵
plt.figure(figsize=(7, 4))
sns.heatmap(cross_tab, annot=True, fmt='d', cmap='YlOrRd')
plt.title('K-Means 군집 vs 실제 상권 구분')
plt.ylabel('K-Means 군집')
plt.tight_layout()
plt.savefig('plot_kmeans_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 군집별 주요 feature 평균 (원래 스케일)
df_feat_orig = df.copy()
df_feat_orig['kmeans_label'] = km_labels
df_feat_orig = df_feat_orig[df_feat_orig['당월_매출_금액'] > 0].reset_index(drop=True)

key_cols = ['당월_매출_금액', '주중_매출_비율', '주말_매출_비율',
            '여성_매출_비율', '연령대_20_비율', '연령대_30_비율',
            '연령대_50_비율', '시간대_17~21_비율', '시간대_21~24_비율',
            '점포_수', '프랜차이즈_비율']

# df_feat에 있는 비율 컬럼 사용
cluster_profile = df_feat.groupby('kmeans_label')[FEATURE_COLS[:15]].mean().round(3)
print('\n=== K-Means 군집별 feature 평균 (표준화 전) ===')
print(cluster_profile.T.to_string())

---
## 결과 요약

| 알고리즘 | 주요 하이퍼파라미터 | 군집 수 | Silhouette Score |
|---|---|---|---|
| K-Means | K=4 | 4 | (결과 기입) |
| Hierarchical | linkage=ward | 4 | (결과 기입) |
| DBSCAN | eps=2.5, min_samples=5 | (결과 기입) | (결과 기입) |
| Affinity Propagation | damping=0.9 | (결과 기입) | (결과 기입) |

### 해석
- PCA / t-SNE 시각화에서 **발달상권**과 **관광특구**는 뚜렷한 분리, **골목상권**과 **전통시장**은 일부 겹침
- K-Means 군집이 실제 label과 가장 잘 일치 → 상권 유형은 매출 패턴(시간대·요일·연령대)으로 구분 가능